# California o9 — statewide reads and the overview pyramid

Reads `california_tdigest_o9.zarr` (written in `02_write`): whole-store discovery, then the #358 pyramid retrofit + a worker-side `mode="sweep"` overview fold — timed, to get the #352 baseline — then the resolution nodes through moczarr's DataTree.

In [ ]:
# %pip install "moczarr>=0.4.0" "zagg[catalog,viz]"

import json
import os
import time
from pathlib import Path

import boto3
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from botocore.config import Config
from matplotlib.colors import LogNorm
from moczarr.convention import decimal_order, morton_decimal
from moczarr.hhdc import read_tensors
from moczarr.open import open_hive, open_leaf, open_store

from zagg.config import default_config
from zagg.sweep_overview import declare_pyramid

In [ ]:
# prefer the 'nasa' profile when it exists (laptop); ambient creds otherwise (hub role)
import botocore.session as _bs

if "nasa" in (_bs.Session().full_config.get("profiles") or {}):
    os.environ.setdefault("AWS_PROFILE", "nasa")

STORE = "s3://sliderule-public/zagg-demo"
CA_STORE = f"{STORE}/california_tdigest_o9.zarr"
OUT = Path("outputs")

timings = {}


class stage:
    def __init__(self, name):
        self.name = name

    def __enter__(self):
        self.t0 = time.perf_counter()
        return self

    def __exit__(self, *exc):
        timings[self.name] = round(time.perf_counter() - self.t0, 2)
        print(f"[{self.name}] {timings[self.name]:.1f}s")

## The whole state, one call

Discovery is lazy — metadata only, zero digest bytes. The `cells` dimension is the **addressable coverage** (every order-19 cell of every committed leaf, from the coverage MOC), not cells with data; the sampled subset lives inside the arrays as non-fill values, and the occupancy mask is what distinguishes observed-empty from never-observed.

In [ ]:
with stage("open_hive: California o9"):
    ca = open_hive(CA_STORE)
ca

## Declare the pyramid (the #358 retrofit)

The store was born `pyramid: false`; `declare_pyramid` installs the overview schedule on the existing manifest from the same config grammar template time uses — declaration is no longer birth-only.

In [ ]:
pyr_config = default_config("atl03_tdigest_strata_healpix")
pyr_config.output = default_config("atl03_tdigest_healpix_hive").output
pyr_config.output["pyramid"] = {"spacing": 2}  # every-2-orders below the o9 shard order

with stage("declare_pyramid (manifest RMW)"):
    block = declare_pyramid(CA_STORE, pyr_config)
block

## Fire the sweep (worker-side, `mode="sweep"`)

One synchronous worker invoke folds leaf artifacts up-tree for every family — including the newly declared overviews across 2,721 leaves' strata digests. This is the #352 baseline measurement: the response carries the #354 per-family timings. It may hit the 900 s worker cap — that outcome is data too.

In [ ]:
lam = boto3.client(
    "lambda", region_name="us-west-2",
    config=Config(read_timeout=960, connect_timeout=10, retries={"max_attempts": 0}),
)
with stage("mode=sweep: full-store fold"):
    resp = lam.invoke(
        FunctionName="process-shard-4096-disk",
        InvocationType="RequestResponse",
        Payload=json.dumps({"mode": "sweep", "store_path": CA_STORE, "discover": True}),
    )
payload = json.loads(resp["Payload"].read())
body = json.loads(payload["body"]) if isinstance(payload.get("body"), str) else payload
body

## The pyramid in the tree

moczarr's `open_store` discovers resolution nodes straight from the manifest declaration (spec §4.5) — the overview zarrs appear as interior nodes of the DataTree.

In [ ]:
with stage("open_store: DataTree with resolution nodes"):
    tree = open_store(CA_STORE)
tree

In [ ]:
# group every node dataset by its cell order; pick the coarsest overview level
by_order = {}
for node in tree.subtree:
    ds_n = getattr(node, "ds", None)
    if ds_n is None or "morton" not in ds_n or not ds_n.sizes.get("cells"):
        continue
    o = decimal_order(morton_decimal(int(ds_n["morton"][0].values)))
    by_order.setdefault(o, []).append(ds_n)
print({o: len(v) for o, v in sorted(by_order.items())})

In [ ]:
# conservation check + statewide density map from an overview level
import xarray as xr

overview_order = min(o for o in by_order if o < 19)
ov = xr.concat([d[["count"]].load() for d in by_order[overview_order]], dim="cells")
print(f"order-{overview_order} overview: {ov.sizes['cells']} cells; "
      f"count sum {int(ov['count'].sum()):,} (leaves: exactness check vs the run total)")

from mortie import mort2polygon

words = [int(w) for w in ov["morton"].values]
lat = np.array([np.mean([p[0] for p in mort2polygon(w, step=4)]) for w in words])
lon = np.array([np.mean([p[1] for p in mort2polygon(w, step=4)]) for w in words])

fig, ax = plt.subplots(figsize=(6.5, 7))
sc = ax.scatter(lon, lat, c=ov["count"].values, s=140, marker="s", cmap="magma",
                norm=LogNorm(vmin=max(1, ov["count"].values.min())))
ax.set_xlabel("lon")
ax.set_ylabel("lat")
ax.set_title(f"California photon density — order-{overview_order} overview", fontsize=11)
fig.colorbar(sc, ax=ax, shrink=0.8, label="photons / cell")
ax.set_aspect("equal")

## A Sierra-crest HHDC block

The monsters from the o8 campaign, read back as ordinary tensors: `open_leaf` + a span-restricted `read_tensors` on one o9 shard at the crest.

In [ ]:
# one of the o8 campaign's border-overhang regions, now an ordinary o9 shard
from moczarr.convention import morton_word

sierra_shard = "323211123"  # the first o8 OOM's neighborhood, o9-sized
try:
    store9 = open_leaf(CA_STORE, sierra_shard)
except Exception:
    # fall back to the densest leaf in the coverage MOC if that label is absent
    sierra_shard = morton_decimal(int(ca["morton"][0].values))[:10]
    store9 = open_leaf(CA_STORE, sierra_shard)

with stage("Sierra shard: signal tensors"):
    blocks_ca = list(
        read_tensors(store9, "19/h_tdigest_signal", n_bins=64, resolution=1.0,
                     block_order=12, fit="degrade_resolution")
    )
print(f"{len(blocks_ca)} blocks; densest: {max(int(b[0].sum()) for b in blocks_ca):,} photons")

In [ ]:
t, m, (off, g), w = max(blocks_ca, key=lambda b: b[0].sum())


def percentile_height(tensor, mask, offset, gain, q):
    counts = tensor.astype("float64")
    total = counts.sum(axis=2)
    idx = (counts.cumsum(axis=2) >= q * total[..., None]).argmax(axis=2)
    return np.where((mask == 2) & (total > 0), offset + (idx + 0.5) * gain, np.nan)


fig, axes = plt.subplots(1, 3, figsize=(12, 3.8), sharey=True)
for ax, q, label in zip(axes, (0.05, 0.50, 0.98), ("terrain (p05)", "median", "surface (p98)")):
    im = ax.imshow(percentile_height(t, m, off, g, q), origin="lower", cmap="viridis")
    ax.set_title(label, fontsize=10)
    ax.set_xticks([]), ax.set_yticks([])
fig.colorbar(im, ax=axes, shrink=0.8, label="elevation (m)")
fig.suptitle(f"block {morton_decimal(w)} — Sierra crest, {g:g} m bins", y=1.02, fontsize=11)

## Timings

In [ ]:
(OUT / "timings_california_read.json").write_text(json.dumps(timings, indent=2))
pd.Series(timings, name="seconds").to_frame()